<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Özlem Salehi Köken
        </td>        
</tr></table>

# Variational Quantum Eigensolver - Basics and Motivation



The Variational Quantum Eigensolver (VQE) is a quantum algorithm designed to approximate (find an upper bound) the *ground-state energy* of a quantum system. It was proposed by Peruzzo et al. in 2014. In this notebook, we will learn about the basics of it and the motivation behind.



## How VQE works?

To answer this question, we need to talk about some concepts like ground state, Hamiltonian and variational principle.


### What is the ground state?

A quantum system can exist in multiple energy states, each associated with a specific energy level. The ground state is the lowest energy state of the system. It is the most stable configuration because, unless external energy is applied, a system will always tend to settle in its ground state.

Ground state energy corresponds to the lowest eigenvalue of the *Hamiltonian* that represents the system.




### What is Hamiltonian?

In quantum mechanics, the Hamiltonian is a matrix that represents the total energy of a system. It includes both the kinetic energy (motion of particles) and potential energy (interaction forces). Its eigenvalues correspond to the possible energy levels of the system which are "allowed", and the corresponding eigenstates are the quantum states that have definite energy values.

The Hamiltonian $H$ must be a *Hermitian* operator, meaning:
$$
H^\dagger = H,
$$
where $H^\dagger$ is the conjugate transpose (or adjoint) of the operator $H$.

This condition ensures that the Hamiltonian has real eigenvalues, which correspond to measurable energy values.





### Why are we interested in the ground state energy?

The ground-state energy is crucial in many fields, helping predicting chemical reactions, analyzing material properties, and improving drug discovery.

However, calculating the ground-state energy of complex molecules using classical computers is computationally expensive.

## Hamiltonians in action

Let us now inspect in more detail what a Hamiltonian looks like.

The Pauli $Z$ matrix represents a single-qubit Hamiltonian that can describe the energy associated with the spin of a qubit along the $z$-axis.

$$
\sigma_Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}.
$$

It is Hermitian and its lowest eigenvalue is -1 with the corresponding eigenstate
$
\begin{pmatrix} 0 \\ 1  \end{pmatrix}
$.

Pauli matrices form a complete set of operators for qubits, meaning that any operator acting on a qubit system can be expressed as a linear combination of these Pauli operators. Hence, a general Hamiltonian for a quantum system can be written as a sum of terms called *Pauli strings*, each of which involves Pauli matrices acting on one or more qubits.

As an example, we can define the following Hamiltonian on two qubits: 

$$
X_0\otimes X_1+Y_0\otimes Y_1+Z_0\otimes Z_1
$$, 

typically written as $XX+YY+ZZ$. In this example, $XX$, $YY$ and $ZZ$ are Pauli strings. This Hamiltonian is known as the Heisenberg Hamiltonian. Its matrix representation can be obtained through the matrices of Pauli $X$, $Y$, and $Z$ operators.

Here are some other random Hamiltonians: $ 2X I X - Y ZI,~~ 3X Y Y + 2Z Z I - 4Y X Y, ~~ -X I  + Y Z + 5X X $. The coefficients represent the strength of the interaction between different Pauli operators acting on the qubits, with both positive and negative values indicating different types of interactions (attractive or repulsive) between qubits.






### Creating Hamiltonians in Qiskit

In Qiskit, one way to create a Hamiltonian consisting of Pauli strings is to use the `Operator` class and `from_label` function. Here is `ZZ` represented as an `Operator`.

In [5]:
from qiskit.quantum_info import Operator

H = Operator.from_label("ZZ")
print(H)

Operator([[ 1.+0.j,  0.+0.j,  0.+0.j,  0.+0.j],
          [ 0.+0.j, -1.+0.j,  0.+0.j,  0.+0.j],
          [ 0.+0.j,  0.+0.j, -1.+0.j,  0.+0.j],
          [ 0.+0.j,  0.+0.j,  0.+0.j,  1.+0.j]],
         input_dims=(2, 2), output_dims=(2, 2))


Operators can be combined using linear operators like addition, subtraction and scalar multiplication. Here is the Hamiltonian $2XIX - YZI$.

In [18]:
from qiskit.quantum_info import Operator

XIX = Operator.from_label("XIX")
YZI = Operator.from_label("YZI")
H = XIX-YZI
print(H)

Operator([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+1.j, 1.+0.j, 0.+0.j, 0.+0.j],
          [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j, 0.+1.j, 0.+0.j, 0.+0.j],
          [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.-1.j, 1.+0.j],
          [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j, 0.-1.j],
          [0.-1.j, 1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
          [1.+0.j, 0.-1.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
          [0.+0.j, 0.+0.j, 0.+1.j, 1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
          [0.+0.j, 0.+0.j, 1.+0.j, 0.+1.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]],
         input_dims=(2, 2, 2), output_dims=(2, 2, 2))


You can get the matrix representation of an Operator as follows:

In [20]:
H.data

array([[0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+1.j, 1.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j, 0.+1.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.-1.j, 1.+0.j],
       [0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 1.+0.j, 0.-1.j],
       [0.-1.j, 1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [1.+0.j, 0.-1.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 0.+1.j, 1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],
       [0.+0.j, 0.+0.j, 1.+0.j, 0.+1.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j]])

### Task 1

Create the Hamiltonian $3X Y Y + 2Z Z I - 4Y X Y$ in Qiskit using `Operator` class. Verify that it is Hermitian.

In [ ]:
from qiskit.quantum_info import Operator
import numpy as np

# Your code here

print(H)

[click for our solution](02_variational_quantum_eigensolver_basics.ipynb#task1)

A more efficient alternative when dealing with large number of qubits is the `SparsePauliOp` class. This object takes a list of Pauli terms and coefficients as input. You can convert it to an `Operator` object or get its matrix representation.

In [9]:
from qiskit.quantum_info import SparsePauliOp

op = SparsePauliOp(["XIX", "YZI"], [1.0, - 1.0])
print(op)
print(op.to_matrix())
print(type(op.to_operator()))

SparsePauliOp(['XIX', 'YZI'],
              coeffs=[ 1.+0.j, -1.+0.j])
[[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+1.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+1.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.-1.j 1.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.-1.j]
 [0.-1.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.-1.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+1.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+1.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]]
<class 'qiskit.quantum_info.operators.operator.Operator'>


You can also sum those objects and then use the `simplify` function to combine duplicates and remove zeros.

In [15]:
H = op + op
print(H)
print(H.simplify())

SparsePauliOp(['XIX', 'YZI', 'XIX', 'YZI'],
              coeffs=[ 1.+0.j, -1.+0.j,  1.+0.j, -1.+0.j])
SparsePauliOp(['XIX', 'YZI'],
              coeffs=[ 2.+0.j, -2.+0.j])


### Task 2

Create the Hamiltonian $3X Y Y + 2Z Z I - 4Y X Y$ in Qiskit using `SparsePauliOp` class.


In [ ]:
from qiskit.quantum_info import SparsePauliOp

# Your code here

print(H)

[click for our solution](02_variational_quantum_eigensolver_basics.ipynb#task2)

### Task 3

Complete the following function that creates a random Hamiltonian given the number of qubits and number of terms. Coefficients should be between -1 and 1. Verify that it is Hermitian.

In [ ]:
from qiskit.quantum_info import SparsePauliOp

def random_hamiltonian(n_qubits, n_terms):
    
    # Your code here


    return H


[click for our solution](02_variational_quantum_eigensolver_basics.ipynb#task3)

### Finding the ground state energy

Ground state energy of a Hamiltonian is its smallest eigenvalue. For large matrices, it is not possible to find this efficiently and this is why VQE is proposed as an alternative method to find an upper bound on the smallest eigenvalue.

Nevertheless, For small Hermitian matrices, we can use `np.linalg.eigvalsh` to compute the eigenvalues.

### Task 4

Compute the eigenvalues of Pauli Z operator.

In [ ]:
import numpy as np

# Your code here

[click for our solution](02_variational_quantum_eigensolver_basics.ipynb#task4)

If you also want to compute the ground state, then you need to find the eigenvectors. For small Hermitian matrices, we can use `np.linalg.eigh` function, which returns eigenvalues and eigenvectors as a tuple.

### Task 5

Create a random Hamiltonian acting on 4 qubits with 5 terms. Find its ground state energy and ground state. Verify that all of its eigenvalues are real. (You may need to is `np.isclose` to check if the imaginary part is 0)

In [ ]:
import numpy as np

# Your code here

print("Ground state energy:", ground_state_energy)
print("Ground state:", ground_state)

[click for our solution](02_variational_quantum_eigensolver_basics.ipynb#task5)